In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 2.3 Train and Compare Regularized Models
## Model Cycle Step 2: **Train**
- Train L2, L1, ElasticNet models
- Compare coefficients across types
- See which features L1 selects
- Cross-validate for performance comparison

## The Model Cycle
### 1. Build
### **2. Train** ← We are here
### 3. Predict
### 4. Evaluate
### 5. Improve

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import pickle
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score

pd.options.display.max_columns = None

In [ ]:
data_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
models_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

df_training = pd.read_csv(f'{data_filepath}training.csv')
X_train = df_training
y_train = df_training['SEM_3_STATUS']
print(f"Training data: {X_train.shape[0]} samples")

## Load the Models We Built

In [ ]:
l2_model = pickle.load(open(f'{models_filepath}l2_ridge_logistic_model.pkl', 'rb'))
l1_model = pickle.load(open(f'{models_filepath}l1_lasso_logistic_model.pkl', 'rb'))
elasticnet_model = pickle.load(open(f'{models_filepath}elasticnet_logistic_model.pkl', 'rb'))
baseline_model = pickle.load(open(f'{models_filepath}baseline_logistic_model.pkl', 'rb'))

models = {
    'Baseline (No Penalty)': baseline_model,
    'L2 (Ridge)': l2_model,
    'L1 (Lasso)': l1_model,
    'ElasticNet': elasticnet_model
}

print("All models loaded successfully!")

## Train All Models

In [ ]:
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print("Done!")

print("\nAll models trained successfully!")

## Compare Coefficients
- How does regularization change coefficient values?
- Which features does L1 zero out?

In [ ]:
preprocessor = trained_models['Baseline (No Penalty)'].named_steps['preprocessing']
feature_names = preprocessor.get_feature_names_out()
feature_names_clean = [name.split('__')[-1] for name in feature_names]
print(f"Number of features: {len(feature_names_clean)}")
print(f"\nFeatures: {feature_names_clean}")

In [ ]:
coef_data = []
for name, model in trained_models.items():
    classifier = model.named_steps['classifier']
    coefficients = classifier.coef_[0]
    for feat, coef in zip(feature_names_clean, coefficients):
        coef_data.append({'Model': name, 'Feature': feat, 'Coefficient': coef})

coef_df = pd.DataFrame(coef_data)

fig = px.bar(coef_df, x='Feature', y='Coefficient', color='Model', barmode='group',
             title='Coefficient Comparison Across Regularization Types', height=500)
fig.update_xaxes(tickangle=45)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02))
fig.show()

## What We See
- **Baseline** (no penalty) → largest coefficients
- **L2** → shrinks proportionally
- **L1** → some coefficients go to exactly zero
- **ElasticNet** → combines both behaviors

In [ ]:
coef_norms = {}
for name, model in trained_models.items():
    classifier = model.named_steps['classifier']
    coefficients = classifier.coef_[0]
    l2_norm = np.sqrt(np.sum(coefficients**2))
    l1_norm = np.sum(np.abs(coefficients))
    coef_norms[name] = {'L2 Norm': l2_norm, 'L1 Norm': l1_norm}

norms_df = pd.DataFrame(coef_norms).T
print("Coefficient Norms (measure of model complexity):")
display(norms_df)

## L1 Feature Selection
- Which features did Lasso keep?
- Which did it eliminate?

In [ ]:
l1_classifier = trained_models['L1 (Lasso)'].named_steps['classifier']
l1_coefficients = l1_classifier.coef_[0]

l1_selection = pd.DataFrame({
    'Feature': feature_names_clean,
    'Coefficient': l1_coefficients,
    'Selected': l1_coefficients != 0
}).sort_values('Coefficient', key=abs, ascending=False)

print("L1 (Lasso) Feature Selection:")
print(f"Total features: {len(l1_selection)}")
print(f"Selected (non-zero): {l1_selection['Selected'].sum()}")
print(f"Eliminated (zero): {(~l1_selection['Selected']).sum()}")
print("\nFeatures by importance:")
display(l1_selection)

In [ ]:
fig = go.Figure()
colors = ['green' if s else 'lightgray' for s in l1_selection['Selected']]
fig.add_trace(go.Bar(x=l1_selection['Feature'], y=l1_selection['Coefficient'], marker_color=colors))
fig.update_layout(title='L1 (Lasso) Coefficients: Green = Selected, Gray = Eliminated',
                  xaxis_tickangle=45, height=400)
fig.show()

## Cross-Validation Comparison
- 5-fold stratified cross-validation
- Optimize for F1 on minority class ('N' = students who leave)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scorer = make_scorer(f1_score, pos_label='N')
precision_scorer = make_scorer(precision_score, pos_label='N')
recall_scorer = make_scorer(recall_score, pos_label='N')

In [ ]:
cv_results = []
for name, model in models.items():
    print(f"Cross-validating {name}...", end=" ")
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=f1_scorer)
    precision_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=precision_scorer)
    recall_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=recall_scorer)
    cv_results.append({
        'Model': name,
        'F1 Mean': f1_scores.mean(), 'F1 Std': f1_scores.std(),
        'Precision Mean': precision_scores.mean(), 'Precision Std': precision_scores.std(),
        'Recall Mean': recall_scores.mean(), 'Recall Std': recall_scores.std()
    })
    print("Done!")

cv_df = pd.DataFrame(cv_results)
print("\nCross-Validation Results (Positive class: 'N' - Students who leave):")
display(cv_df)

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=('F1 Score', 'Precision', 'Recall'))
for i, metric in enumerate(['F1', 'Precision', 'Recall'], 1):
    fig.add_trace(go.Bar(x=cv_df['Model'], y=cv_df[f'{metric} Mean'],
                         error_y=dict(type='data', array=cv_df[f'{metric} Std']),
                         name=metric, showlegend=False), row=1, col=i)
fig.update_layout(height=400, title_text='Cross-Validation Results by Regularization Type')
fig.update_xaxes(tickangle=45)
fig.show()

## Save Trained Models

In [ ]:
for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    filepath = f'{models_filepath}{filename}_trained.pkl'
    pickle.dump(model, open(filepath, 'wb'))
    print(f"Saved: {filepath}")

## Key Takeaways
- Regularization reduces coefficient magnitudes vs baseline
- L1 (Lasso) naturally performs feature selection
- Regularized models often show similar or improved CV performance
- Cross-validation helps us compare fairly

**Next:** 2.4 Tune Regularization Hyperparameters